## M4 — Clustering K-Means
Segmentamos las 100 ubicaciones en perfiles para enriquecer la narrativa ante el jurado.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('top100_ubicaciones.csv', encoding='utf-8-sig')

print(f"Ubicaciones cargadas: {df.shape[0]}")
print(f"Columnas: {df.columns.tolist()}")

Ubicaciones cargadas: 100
Columnas: ['CPRO', 'CMUN', 'NOMBRE', 'secciones', 'fofm_medio', 'fofm_max', 'vol_75plus_total', 'pct_75plus_media', 'ratio_viudez_media', 'cohorte_futura', 'renta_hogar', 'latitud', 'longitud', 'ranking_municipio', 'posicion_final']


### Paso 1 — Seleccionar variables y escalar

In [2]:
# Variables que definen el perfil de cada municipio
features = [
    'renta_hogar',
    'vol_75plus_total',
    'pct_75plus_media',
    'ratio_viudez_media',
    'cohorte_futura'
]

X = df[features].copy()

# Escalar — media 0, desviación estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables escaladas:")
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

Variables escaladas:
       renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  \
count       100.00            100.00            100.00              100.00   
mean         -0.00              0.00              0.00                0.00   
std           1.01              1.01              1.01                1.01   
min          -1.00             -0.56             -1.97               -1.92   
25%          -0.56             -0.54             -0.65               -0.82   
50%          -0.29             -0.46             -0.02                0.08   
75%           0.12              0.06              0.57                0.66   
max           5.34              6.20              3.14                2.90   

       cohorte_futura  
count          100.00  
mean             0.00  
std              1.01  
min             -0.59  
25%             -0.55  
50%             -0.44  
75%              0.10  
max              6.10  


### Paso 2 — Encontrar el número óptimo de clusters

In [3]:
resultados = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels)
    inercia = kmeans.inertia_
    resultados.append({'k': k, 'silhouette': round(sil, 4), 'inercia': round(inercia, 1)})

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))

 k  silhouette  inercia
 2      0.4219    345.7
 3      0.2834    268.0
 4      0.2931    209.6
 5      0.2926    168.4
 6      0.3081    132.5
 7      0.3038    116.4
 8      0.2905    101.0


### Paso 3 — Aplicar K-Means con 4 clusters

In [4]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Ver cuántos municipios hay en cada cluster
print("Municipios por cluster:")
print(df['cluster'].value_counts().sort_index())

# Ver las características medias de cada cluster
perfil = df.groupby('cluster')[features].mean().round(0)
print("\nPerfil medio por cluster:")
print(perfil.to_string())

Municipios por cluster:
cluster
0    25
1    15
2     6
3    54
Name: count, dtype: int64

Perfil medio por cluster:
         renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  cohorte_futura
cluster                                                                                     
0            29011.0            2453.0              15.0                 2.0          2185.0
1            31712.0           27916.0              13.0                 2.0         27771.0
2            65148.0            3876.0               8.0                 2.0          4756.0
3            34747.0            3007.0               9.0                 2.0          3314.0


### Paso 4 — Etiquetar los clusters con nombres estratégicos

In [5]:
nombres_cluster = {
    0: "Zona envejecida — solvencia ajustada",
    1: "Gran ciudad — mercado masivo",
    2: "Zona premium — máxima solvencia",
    3: "Municipio equilibrado — perfil estándar"
}

df['perfil'] = df['cluster'].map(nombres_cluster)

# Resumen por perfil
resumen = df.groupby('perfil').agg(
    municipios      = ('NOMBRE', 'count'),
    renta_media     = ('renta_hogar', 'mean'),
    vol_75plus      = ('vol_75plus_total', 'mean'),
    fofm_medio      = ('fofm_medio', 'mean')
).round(0)

print("=== PERFILES ESTRATÉGICOS ===")
print(resumen.to_string())

# Ver qué municipios hay en cada perfil
for perfil in nombres_cluster.values():
    municipios = df[df['perfil'] == perfil]['NOMBRE'].tolist()
    print(f"\n{perfil}:")
    print(", ".join(municipios[:10]))
    if len(municipios) > 10:
        print(f"  ... y {len(municipios)-10} más")

=== PERFILES ESTRATÉGICOS ===
                                         municipios  renta_media  vol_75plus  fofm_medio
perfil                                                                                  
Gran ciudad — mercado masivo                     15      31712.0     27916.0         0.0
Municipio equilibrado — perfil estándar          54      34747.0      3007.0         0.0
Zona envejecida — solvencia ajustada             25      29011.0      2453.0         0.0
Zona premium — máxima solvencia                   6      65148.0      3876.0         0.0

Zona envejecida — solvencia ajustada:
Pontes de García Rodríguez, As, Bergondo, Laudio/Llodio, Cervo, Cangas del Narcea, Pola de Gordón, La, Coaña, Xove, Segovia, Corgo, O
  ... y 15 más

Gran ciudad — mercado masivo:
Donostia/San Sebastián, Pamplona/Iruña, Santander, Vitoria-Gasteiz, Burgos, León, Oviedo, Valladolid, Zaragoza, Vigo
  ... y 5 más

Zona premium — máxima solvencia:
Pozuelo de Alarcón, Matadepera, Boadilla del Monte, 

### Paso 5 — Guardar resultados del clustering

In [7]:
# Guardar top100 con cluster asignado
df.to_csv('top100_con_clusters.csv', index=False, encoding='utf-8-sig')

# Excel actualizado con la columna de perfil
df_muni = pd.read_csv('ranking_municipios.csv', encoding='utf-8-sig')
resumen_prov = pd.read_csv('resumen_provincias.csv', encoding='utf-8-sig')

with pd.ExcelWriter('resultados_akademia.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_muni.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_prov.to_excel(writer, sheet_name='Resumen por provincia', index=False)
    
    # Nueva pestaña con resumen de perfiles
    resumen_clusters = df.groupby('perfil').agg(
        municipios  = ('NOMBRE', 'count'),
        renta_media = ('renta_hogar', 'mean'),
        vol_75plus  = ('vol_75plus_total', 'mean'),
        fofm_medio  = ('fofm_medio', 'mean')
    ).round(2).reset_index()
    resumen_clusters.to_excel(writer, sheet_name='Perfiles estratégicos', index=False)

print("Archivos guardados:")
print(f"  top100_con_clusters.csv → {len(df)} municipios con perfil asignado")
print(f"  resultados_akademia.xlsx → actualizado con pestaña de perfiles")

Archivos guardados:
  top100_con_clusters.csv → 100 municipios con perfil asignado
  resultados_akademia.xlsx → actualizado con pestaña de perfiles
